## **02. Nettoyage et preprocessing**

In [34]:
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
 
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11
})

walmart_store_sales_data = '../data/Walmart_Store_sales.csv'
output_images_path = '../output/images'
output_data_path = '../output/data'
output_models_path = '../output/models'
output_objects_path = '../output/objects'

# Chargement des données
df = pd.read_csv(walmart_store_sales_data)
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [35]:
df_clean = df.copy()
initial_rows = len(df_clean)

# Supprimer les lignes où Weekly_Sales est manquant
df_clean = df_clean.dropna(subset=['Weekly_Sales'])
print(f"Lignes supprimées (Weekly_Sales manquant) : {initial_rows - len(df_clean)}")
print(f"Lignes restantes : {len(df_clean)}")


Lignes supprimées (Weekly_Sales manquant) : 14
Lignes restantes : 136


In [36]:
# Imputer les valeurs manquantes des features (médiane pour numériques, mode pour catégorielles)
print("Imputation des valeurs manquantes\n")
for col in ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
    n_miss = df_clean[col].isnull().sum()
    if n_miss > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"{col}: {n_miss} valeurs imputées par la médiane ({median_val:.2f})")
 
for col in ['Holiday_Flag', 'Store']:
    n_miss = df_clean[col].isnull().sum()
    if n_miss > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        print(f"\n{col}: {n_miss} valeurs imputées par le mode ({mode_val})")

# Imputer Date manquante (on la supprime car on ne peut pas deviner)
n_date_miss = df_clean['Date'].isnull().sum()
if n_date_miss > 0:
    df_clean = df_clean.dropna(subset=['Date'])
    print(f"\nDate: {n_date_miss} lignes supprimées (date non imputable)")


Imputation des valeurs manquantes

Temperature: 15 valeurs imputées par la médiane (62.25)
Fuel_Price: 12 valeurs imputées par la médiane (3.45)
CPI: 11 valeurs imputées par la médiane (196.92)
Unemployment: 14 valeurs imputées par la médiane (7.48)

Holiday_Flag: 11 valeurs imputées par le mode (0.0)

Date: 18 lignes supprimées (date non imputable)


In [37]:
# Transformation de la colonne Date
print("\nFeature Engineering : Date")

df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%d-%m-%Y')
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Day'] = df_clean['Date'].dt.day
df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek

print(f"Variables créées : Year, Month, Day, DayOfWeek")
print(f"Plage temporelle : {df_clean['Date'].min().date()} au {df_clean['Date'].max().date()}")



Feature Engineering : Date
Variables créées : Year, Month, Day, DayOfWeek
Plage temporelle : 2010-02-05 au 2012-10-19


In [38]:
# Gestion des outliers (méthode 3-sigma)
print("\nSuppression des outliers (3-sigma)\n")

outlier_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
rows_before = len(df_clean)

for col in outlier_cols:
    mean = df_clean[col].mean()
    std = df_clean[col].std()
    lower = mean - 3 * std
    upper = mean + 3 * std
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
    print(f"{col}: {n_out} outliers supprimés")

print(f"\nTotal outliers supprimés : {rows_before - len(df_clean)}")
print(f"Dataset final : {len(df_clean)} lignes")


Suppression des outliers (3-sigma)

Temperature: 0 outliers supprimés
Fuel_Price: 0 outliers supprimés
CPI: 0 outliers supprimés
Unemployment: 5 outliers supprimés

Total outliers supprimés : 5
Dataset final : 113 lignes


In [39]:
# Vérification finale
print("Vérification post-nettoyage\n")
print(f"Valeurs manquantes restantes : {df_clean.isnull().sum().sum()}")
print(df_clean.describe().round(2))

Vérification post-nettoyage

Valeurs manquantes restantes : 0
        Store                        Date  Weekly_Sales  Holiday_Flag  \
count  113.00                         113        113.00        113.00   
mean     9.86  2011-04-24 21:52:33.982301    1267414.77          0.06   
min      1.00         2010-02-05 00:00:00     268929.03          0.00   
25%      4.00         2010-07-30 00:00:00     563460.77          0.00   
50%      9.00         2011-04-22 00:00:00    1420405.41          0.00   
75%     15.00         2012-01-13 00:00:00    1847430.96          0.00   
max     20.00         2012-10-19 00:00:00    2771397.17          1.00   
std      6.18                         NaN     674682.43          0.24   

       Temperature  Fuel_Price     CPI  Unemployment     Year   Month     Day  \
count       113.00      113.00  113.00        113.00   113.00  113.00  113.00   
mean         60.38        3.29  181.44          7.39  2010.83    6.27   16.53   
min          18.79        2.51  126.1

In [43]:
# CONVERSION DES TYPES
df_clean['Store'] = df_clean['Store'].astype(int)
df_clean['Holiday_Flag'] = df_clean['Holiday_Flag'].astype(int)
df_clean['Year'] = df_clean['Year'].astype(int)
df_clean['Month'] = df_clean['Month'].astype(int)
df_clean['Day'] = df_clean['Day'].astype(int)
df_clean['DayOfWeek'] = df_clean['DayOfWeek'].astype(int)

df_clean = df_clean.drop(columns=['Date'])
df_clean.info()

<class 'pandas.DataFrame'>
Index: 113 entries, 0 to 149
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         113 non-null    int64  
 1   Weekly_Sales  113 non-null    float64
 2   Holiday_Flag  113 non-null    int64  
 3   Temperature   113 non-null    float64
 4   Fuel_Price    113 non-null    float64
 5   CPI           113 non-null    float64
 6   Unemployment  113 non-null    float64
 7   Year          113 non-null    int64  
 8   Month         113 non-null    int64  
 9   Day           113 non-null    int64  
 10  DayOfWeek     113 non-null    int64  
dtypes: float64(5), int64(6)
memory usage: 10.6 KB


In [41]:
# VÉRIFICATION DU DATASET NETTOYÉ

print(f"Shape  : {df_clean.shape}")
print(f"NaN restants : {df_clean.isnull().sum().sum()}")
print(f"Doublons     : {df_clean.duplicated().sum()}")
 
print(f"\nStatistiques du dataset nettoyé")
print(df_clean.describe().round(2))


Shape  : (113, 12)
NaN restants : 0
Doublons     : 0

Statistiques du dataset nettoyé
        Store                        Date  Weekly_Sales  Holiday_Flag  \
count  113.00                         113        113.00        113.00   
mean     9.86  2011-04-24 21:52:33.982301    1267414.77          0.06   
min      1.00         2010-02-05 00:00:00     268929.03          0.00   
25%      4.00         2010-07-30 00:00:00     563460.77          0.00   
50%      9.00         2011-04-22 00:00:00    1420405.41          0.00   
75%     15.00         2012-01-13 00:00:00    1847430.96          0.00   
max     20.00         2012-10-19 00:00:00    2771397.17          1.00   
std      6.18                         NaN     674682.43          0.24   

       Temperature  Fuel_Price     CPI  Unemployment     Year   Month     Day  \
count       113.00      113.00  113.00        113.00   113.00  113.00  113.00   
mean         60.38        3.29  181.44          7.39  2010.83    6.27   16.53   
min          

In [42]:
# Sauvegarde des données
print("Sauvegarde des données\n")
df_clean.to_csv(f"{output_data_path}/df_preprocessed.csv", index=False)
print(f"Fichiers sauvegardés dans {output_data_path}")

Sauvegarde des données

Fichiers sauvegardés dans ../output/data
